# M11.7 — Streaming G0/G1/R domain generation

## Purpose

M11.6 established and validated the physical construction of the three
controlled noise domains:

$$
G0 =
h(\theta,d_{G0})
+
n_{\rm Gaussian}[S_n^{\rm analytical}],
$$

$$
G1 =
h(\theta,d_{\rm emp})
+
n_{\rm Gaussian}[S_n^{\rm empirical}],
$$

$$
R =
h(\theta,d_{\rm emp})
+
n_{\rm real}.
$$

The objective of this notebook is **not** to revalidate the physics.

Its purpose is to validate the production strategy required for the full
M11.6 dataset:

1. load the frozen master manifest;
2. process only one real-noise `file_group_id` at a time;
3. keep only local PSDs and one HLV strain environment in memory;
4. generate paired G0/G1/R examples;
5. write each example immediately to disk;
6. release the real strain before moving to the next environment.

This avoids the cumulative real-strain cache used during C3 validation,
which is unsuitable for production because of its memory cost.

The first test uses only one `file_group_id` and five sources.

No training dataset is produced yet.

In [1]:
from pathlib import Path
import json
import gc

import h5py
import numpy as np
import pandas as pd

In [2]:
DATA_ROOT = Path(
    "/data/vserrano/cbc_pe_data"
)

MANIFEST_PATH = (
    DATA_ROOT
    / "processed"
    / "m11_6_manifests"
    / "m11_6_master_experiment_manifest.csv"
)

OUTPUT_DIR = (
    DATA_ROOT
    / "processed"
    / "m11_7_streaming_pilot"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Manifest:")
print(MANIFEST_PATH)

print("\nPilot output:")
print(OUTPUT_DIR)

Manifest:
/data/vserrano/cbc_pe_data/processed/m11_6_manifests/m11_6_master_experiment_manifest.csv

Pilot output:
/data/vserrano/cbc_pe_data/processed/m11_7_streaming_pilot


In [3]:
df_manifest = pd.read_csv(
    MANIFEST_PATH
)

print(
    "Manifest rows:",
    len(df_manifest),
)

print(
    "Splits:",
    df_manifest["split"]
    .value_counts()
    .to_dict(),
)

print(
    "Unique file groups:",
    df_manifest[
        "file_group_id"
    ].nunique(),
)

Manifest rows: 26000
Splits: {'train': 20000, 'val': 2000, 'cal': 2000, 'test': 2000}
Unique file groups: 29


preservar el índice maestro

Esto es importante porque vamos a procesar por file_group_id, pero queremos que los HDF5 mantengan el orden global de las fuentes.

Conceptualmente:

G0[i]↔G1[i]↔R[i].

In [4]:
df_manifest = (
    df_manifest
    .reset_index(drop=True)
    .copy()
)

df_manifest[
    "manifest_index"
] = np.arange(
    len(df_manifest),
    dtype=np.int64,
)

assert (
    df_manifest[
        "manifest_index"
    ].is_unique
)

assert (
    df_manifest[
        "source_id"
    ].is_unique
)

print(
    "Manifest indexing contract passed."
)

Manifest indexing contract passed.


In [5]:
df_train = (
    df_manifest[
        df_manifest["split"]
        == "train"
    ]
)

group_counts = (
    df_train[
        "file_group_id"
    ]
    .value_counts()
)

display(
    group_counts
    .sort_index()
    .rename("n_sources")
    .to_frame()
)

,n_sources
file_group_id,
0,1018
2,970
3,1044
4,1003
6,1062
9,1006
10,1025
11,1017
14,1034


In [6]:
PILOT_FILE_GROUP_ID = int(
    group_counts.index[0]
)

print(
    "Pilot file_group_id:",
    PILOT_FILE_GROUP_ID,
)

print(
    "Available train sources:",
    int(
        group_counts.iloc[0]
    ),
)

Pilot file_group_id: 6
Available train sources: 1062


In [7]:
df_pilot_group = (
    df_train[
        df_train[
            "file_group_id"
        ]
        == PILOT_FILE_GROUP_ID
    ]
    .copy()
    .sort_values(
        "chirp_mass"
    )
    .reset_index(drop=True)
)

pilot_positions = np.linspace(
    0,
    len(df_pilot_group) - 1,
    5,
    dtype=int,
)

df_pilot = (
    df_pilot_group
    .iloc[
        pilot_positions
    ]
    .copy()
    .reset_index(drop=True)
)

display(
    df_pilot[
        [
            "manifest_index",
            "source_id",
            "split",
            "chirp_mass",
            "total_mass",
            "chi_eff",
            "target_network_snr",
            "file_group_id",
            "block_id",
            "noise_crop_id",
        ]
    ]
)

,manifest_index,source_id,split,chirp_mass,total_mass,chi_eff,target_network_snr,file_group_id,block_id,noise_crop_id
0,5444,M116_SRC_005444,train,5.991236,13.938164,-0.276228,18.602381,6,M116_RB_00043,M116_RC_000656
1,10223,M116_SRC_010223,train,24.533322,90.241524,-0.716562,15.458726,6,M116_RB_00043,M116_RC_000649
2,6138,M116_SRC_006138,train,36.800386,89.858622,0.271620,15.247832,6,M116_RB_00043,M116_RC_000646
3,8483,M116_SRC_008483,train,51.215122,118.714474,0.530280,15.877320,6,M116_RB_00043,M116_RC_000648
4,16123,M116_SRC_016123,train,76.068620,174.776175,-0.343616,12.155867,6,M116_RB_00043,M116_RC_000645


In [8]:
assert len(
    df_pilot
) == 5

assert (
    df_pilot[
        "file_group_id"
    ]
    .nunique()
    == 1
)

assert (
    df_pilot[
        "file_group_id"
    ]
    .iloc[0]
    ==
    PILOT_FILE_GROUP_ID
)

assert (
    df_pilot[
        "split"
    ]
    == "train"
).all()

assert (
    df_pilot[
        "source_id"
    ].is_unique
)

print(
    "Pilot selection contracts passed."
)

Pilot selection contracts passed.


## Pilot HDF5 schema

Each domain file stores the final network input and the minimum metadata
required to preserve the paired experimental design.

The primary datasets are

$$
X \in \mathbb{R}^{N\times3\times16384}
$$

and

$$
y =
[M_{\rm chirp},M_{\rm total},\chi_{\rm eff}].
$$

`X` contains the final M10-normalized model input.

Labels remain in physical units. Label standardization is intentionally not
performed during generation; train-only label statistics will be fitted later
by the training pipeline.

The same `manifest_index` and `source_id` identify corresponding examples
across G0, G1 and R.

In [9]:
PILOT_N = len(
    df_pilot
)

pilot_h5_paths = {
    domain:
        OUTPUT_DIR
        / f"m11_7_streaming_pilot_{domain}.h5"

    for domain in [
        "G0",
        "G1",
        "R",
    ]
}

for domain, path in (
    pilot_h5_paths.items()
):
    print(
        domain,
        "->",
        path,
    )

G0 -> /data/vserrano/cbc_pe_data/processed/m11_7_streaming_pilot/m11_7_streaming_pilot_G0.h5
G1 -> /data/vserrano/cbc_pe_data/processed/m11_7_streaming_pilot/m11_7_streaming_pilot_G1.h5
R -> /data/vserrano/cbc_pe_data/processed/m11_7_streaming_pilot/m11_7_streaming_pilot_R.h5


In [10]:
def create_pilot_hdf5(
    path,
    *,
    n_samples,
    domain,
):
    string_dtype = (
        h5py.string_dtype(
            encoding="utf-8"
        )
    )

    with h5py.File(
        path,
        "w",
    ) as h5:

        # -----------------------------------------
        # Primary data
        # -----------------------------------------

        h5.create_dataset(
            "X",
            shape=(
                n_samples,
                3,
                16384,
            ),
            dtype=np.float32,
            chunks=(
                1,
                3,
                16384,
            ),
        )

        h5.create_dataset(
            "y",
            shape=(
                n_samples,
                3,
            ),
            dtype=np.float32,
        )

        # -----------------------------------------
        # Identity / split
        # -----------------------------------------

        h5.create_dataset(
            "source_id",
            shape=(n_samples,),
            dtype=string_dtype,
        )

        h5.create_dataset(
            "manifest_index",
            shape=(n_samples,),
            dtype=np.int64,
        )

        h5.create_dataset(
            "split",
            shape=(n_samples,),
            dtype=string_dtype,
        )

        # -----------------------------------------
        # SNR / distance
        # -----------------------------------------

        for name in [
            "target_network_snr",
            "final_network_snr",
            "distance_mpc",
            "snr_H1",
            "snr_L1",
            "snr_V1",
        ]:

            h5.create_dataset(
                name,
                shape=(n_samples,),
                dtype=np.float64,
            )

        # -----------------------------------------
        # Geometry
        # -----------------------------------------

        for name in [
            "geocentric_time",
            "placement_offset_s",
            "full_network_duration",
            "required_final_duration",
        ]:

            h5.create_dataset(
                name,
                shape=(n_samples,),
                dtype=np.float64,
            )

        # -----------------------------------------
        # Environment
        # -----------------------------------------

        h5.create_dataset(
            "file_group_id",
            shape=(n_samples,),
            dtype=np.int64,
        )

        h5.create_dataset(
            "block_id",
            shape=(n_samples,),
            dtype=string_dtype,
        )

        h5.create_dataset(
            "noise_crop_id",
            shape=(n_samples,),
            dtype=string_dtype,
        )

        h5.create_dataset(
            "is_truncated",
            shape=(n_samples,),
            dtype=np.bool_,
        )

        # 0 = not written
        # 1 = complete
        h5.create_dataset(
            "status",
            data=np.zeros(
                n_samples,
                dtype=np.uint8,
            ),
        )

        # -----------------------------------------
        # File-level metadata
        # -----------------------------------------

        h5.attrs[
            "domain"
        ] = domain

        h5.attrs[
            "sampling_frequency_hz"
        ] = 4096

        h5.attrs[
            "duration_s"
        ] = 4.0

        h5.attrs[
            "detector_order"
        ] = "H1,L1,V1"

        h5.attrs[
            "input_normalization"
        ] = (
            "per_sample_per_detector_zscore"
        )

        h5.attrs[
            "labels"
        ] = (
            "chirp_mass,total_mass,chi_eff"
        )

        h5.attrs[
            "waveform_approximant"
        ] = (
            "SEOBNRv4_opt"
        )

        h5.attrs[
            "low_frequency_cutoff_hz"
        ] = 30.0

In [11]:
for domain, path in (
    pilot_h5_paths.items()
):

    if path.exists():
        path.unlink()

    create_pilot_hdf5(
        path,
        n_samples=PILOT_N,
        domain=domain,
    )

print(
    "Created pilot HDF5 files."
)

Created pilot HDF5 files.


In [12]:
for domain, path in (
    pilot_h5_paths.items()
):

    print(
        "\n",
        "=" * 60,
    )

    print(domain)

    with h5py.File(
        path,
        "r",
    ) as h5:

        for key in h5.keys():

            print(
                f"{key:30s}",
                h5[key].shape,
                h5[key].dtype,
            )


G0
X                              (5, 3, 16384) float32
block_id                       (5,) object
distance_mpc                   (5,) float64
file_group_id                  (5,) int64
final_network_snr              (5,) float64
full_network_duration          (5,) float64
geocentric_time                (5,) float64
is_truncated                   (5,) bool
manifest_index                 (5,) int64
noise_crop_id                  (5,) object
placement_offset_s             (5,) float64
required_final_duration        (5,) float64
snr_H1                         (5,) float64
snr_L1                         (5,) float64
snr_V1                         (5,) float64
source_id                      (5,) object
split                          (5,) object
status                         (5,) uint8
target_network_snr             (5,) float64
y                              (5, 3) float32

G1
X                              (5, 3, 16384) float32
block_id                       (5,) object
distance_mpc      

In [13]:
for domain, path in (
    pilot_h5_paths.items()
):

    size_mb = (
        path.stat().st_size
        /
        1024**2
    )

    print(
        f"{domain}: "
        f"{size_mb:.2f} MB"
    )

G0: 0.01 MB
G1: 0.01 MB
R: 0.01 MB


## M11.7-B — Reuse of the validated M11.6 physical builder

The physical G0/G1/R construction is not reimplemented in this notebook.

The reusable builder and its supporting helpers are copied verbatim from the
validated M11.6 C3 implementation.

M11.7 adds only a different execution strategy:

- one real HLV environment in memory at a time;
- local block-level PSD caching;
- immediate HDF5 writing;
- no accumulation of generated examples in RAM.

Therefore, differences between M11.6 and M11.7 are operational rather than
physical.

In [14]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ks_2samp, wasserstein_distance

from pycbc.types import TimeSeries, FrequencySeries

PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

/afs/ciemat.es/user/v/vserrano/miniconda3/envs/gw-env/lib/python3.10/site-packages/pycbc/types/array.py:36: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal as _lal


In [15]:
from src.config import SimulationConfig

from src.noise import NoiseModel

from src.waveform import WaveformGenerator

from src.detectors import DetectorProjector

from src.windowing import (
    ProjectedNetworkWindowSelector,
)

from src.processing import SignalProcessor

from src.snr import (
    compute_network_optimal_snr,
    rescale_distance_for_target_network_snr,
    validate_snr_rescaling,
)

from pycbc.psd import interpolate
from pycbc.noise import gaussian

from src.injection import SignalInjector
from src.parameters import CBCParameters

from src.models.dataset import (
    normalize_input_per_sample_per_detector_zscore,
)

from src.real_data.catalog import (
    fetch_gwosc_catalog_events,
    gwosc_events_to_parameter_df,
    build_gwosc_urls_for_events,
)

from src.real_data.gwosc_utils import (
    download_if_needed,
    read_gwosc_hdf5_as_pycbc_timeseries,
)

/afs/ciemat.es/user/v/vserrano/miniconda3/envs/gw-env/lib/python3.10/site-packages/pycbc/waveform/plugin.py:99: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [16]:
M116_C2_CONFIG = SimulationConfig(
    simulation_regime="BBH",
    waveform_family="IMR",

    sampling_frequency=4096.0,
    duration=4.0,

    low_frequency_cutoff=30.0,
    waveform_approximant="SEOBNRv4_opt",

    # We control rho* explicitly from the C1 manifest.
    target_network_snr_range=None,

    snr_relative_tolerance=0.05,
    snr_on_truncated_signal=True,

    truncation_policy="keep_last_segment",
    required_final_duration=1.0,

    safe_margin_start=0.0,
    safe_margin_end=0.0,

    processing_context_start_samples=1664,
    processing_context_end_samples=1664,
)

In [17]:
M116_DETECTORS = (
    "H1",
    "L1",
    "V1",
)

M116_M10_INPUT_ZSCORE_EPS = 1e-6

M116_C_MASTER_SEED = 123

M116_CATALOG = "GWTC-3-confident"
M116_SAMPLE_RATE = 4096

M116_C2_PSD_SEGMENT_DURATION_S = 8.0

M116_C3_BUILD_SEED = 24680
M116_C3_PLACEMENT_SEED_OFFSET = 100_000
M116_C3_GAUSSIAN_SEED_OFFSET = 200_000

In [18]:
GWOSC_CACHE_M116 = (
    DATA_ROOT
    / "gwosc_cache"
    / "m11"
)

GWOSC_CACHE_M116.mkdir(
    parents=True,
    exist_ok=True,
)

In [19]:
waveform_generator_M116 = (
    WaveformGenerator(
        config=M116_C2_CONFIG
    )
)

noise_model_G0_M116 = NoiseModel(
    config=M116_C2_CONFIG
)

detector_projector_M116 = (
    DetectorProjector(
        detector_names=list(
            M116_DETECTORS
        )
    )
)

window_selector_M116 = (
    ProjectedNetworkWindowSelector(
        config=M116_C2_CONFIG
    )
)

processor_M116 = SignalProcessor(
    config=M116_C2_CONFIG,

    whitening_method="psd",

    apply_highpass=True,
    apply_lowpass=True,
    apply_standardization=False,

    output_mode="crop_to_config",

    whitening_low_frequency_cutoff=30.0,
    whitening_max_filter_duration=0.5,
    whitening_trunc_method="hann",

    highpass_frequency=30.0,
    lowpass_frequency=512.0,

    fir_order=256,
    fir_beta=5.0,

    remove_corrupted=True,

    rng=np.random.default_rng(
        M116_C_MASTER_SEED + 300
    ),
)

In [20]:
psds_G0_snr_M116 = {
    ifo:
        noise_model_G0_M116.get_psd(
            ifo,
            length=M116_C2_CONFIG.length,
        )
    for ifo in M116_DETECTORS
}

psds_G0_proc_M116 = {
    ifo:
        noise_model_G0_M116.get_psd(
            ifo,
            length=M116_C2_CONFIG.processing_length,
        )
    for ifo in M116_DETECTORS
}

In [21]:
def estimate_raw_offsource_psd_M116(
    strain,
    *,
    psd_start,
    psd_end,
    delta_f,
    target_flength,
    psd_segment_duration=8.0,
):
    psd_data = strain.time_slice(
        float(psd_start),
        float(psd_end),
    )

    if len(psd_data) == 0:
        raise ValueError(
            "PSD reference window is empty."
        )

    if not np.all(
        np.isfinite(
            psd_data.numpy()
        )
    ):
        raise ValueError(
            "PSD reference contains non-finite values."
        )

    # Raw empirical PSD estimate:
    # strain -> Welch -> interpolation.
    #
    # No inverse-spectrum truncation is applied here.
    # Whitening conditioning is handled later inside SignalProcessor.
    psd = psd_data.psd(
        float(psd_segment_duration)
    )

    psd = interpolate(
        psd,
        float(delta_f),
    )

    target_flength = int(
        target_flength
    )

    if len(psd) > target_flength:
        psd = psd[
            :target_flength
        ]

    elif len(psd) < target_flength:
        raise ValueError(
            "Interpolated PSD is shorter than "
            f"target_flength: "
            f"{len(psd)} < {target_flength}"
        )

    if not np.all(
        np.isfinite(
            psd.numpy()
        )
    ):
        raise ValueError(
            "Raw empirical PSD contains "
            "non-finite values."
        )

    return psd

In [22]:
def estimate_block_psds_for_detector_M116(
    *,
    strain,
    psd_start,
    psd_end,
):
    psd_snr = (
        estimate_raw_offsource_psd_M116(
            strain,

            psd_start=psd_start,
            psd_end=psd_end,

            delta_f=(
                M116_C2_CONFIG.delta_f
            ),

            target_flength=(
                M116_C2_CONFIG.flength
            ),

            psd_segment_duration=(
                M116_C2_PSD_SEGMENT_DURATION_S
            ),
        )
    )

    psd_proc = (
        estimate_raw_offsource_psd_M116(
            strain,

            psd_start=psd_start,
            psd_end=psd_end,

            delta_f=(
                M116_C2_CONFIG.processing_delta_f
            ),

            target_flength=(
                M116_C2_CONFIG.processing_flength
            ),

            psd_segment_duration=(
                M116_C2_PSD_SEGMENT_DURATION_S
            ),
        )
    )

    return {
        "snr": psd_snr,
        "proc": psd_proc,
    }

In [23]:
def extract_exact_processing_context_M116(
    strain,
    *,
    start_time,
    expected_length,
):
    dt = float(
        strain.delta_t
    )

    source_start = float(
        strain.start_time
    )

    start_index_float = (
        (
            float(start_time)
            -
            source_start
        )
        /
        dt
    )

    start_index = int(
        round(
            start_index_float
        )
    )

    reconstructed_start = (
        source_start
        +
        start_index
        * dt
    )

    if not np.isclose(
        reconstructed_start,
        float(start_time),
        atol=0.51 * dt,
        rtol=0.0,
    ):
        raise ValueError(
            "Requested processing start is not "
            "aligned with strain sampling grid. "
            f"requested={start_time}, "
            f"reconstructed={reconstructed_start}"
        )

    end_index = (
        start_index
        +
        int(expected_length)
    )

    if (
        start_index < 0
        or
        end_index > len(strain)
    ):
        raise ValueError(
            "Requested processing interval "
            "outside available strain."
        )

    out = (
        strain[
            start_index:end_index
        ]
        .copy()
    )

    out.start_time = (
        reconstructed_start
    )

    return out

In [24]:
def build_signal_network_for_params_c3_M116(
    params,
    *,
    geocentric_time,
    final_start,
    injector,
):
    waveform = (
        waveform_generator_M116.generate(
            params
        )
    )

    projection = (
        detector_projector_M116.project(
            h_plus=waveform.h_plus,
            h_cross=waveform.h_cross,
            parameters=params,
            geocentric_coalescence_time=(
                geocentric_time
            ),
        )
    )

    windowed = (
        window_selector_M116.select(
            projected_strains=(
                projection.strains
            ),
            max_duration=(
                M116_C2_CONFIG.duration
            ),
        )
    )

    zeros = {
        ifo:
            injector.build_zero_strain(
                start_time=final_start,
                length=(
                    M116_C2_CONFIG.length
                ),
            )

        for ifo in M116_DETECTORS
    }

    injected = (
        injector.inject_network(
            noises=zeros,
            signals=(
                windowed.strains
            ),
        )
    )

    segments = {
        ifo:
            injected[ifo].strain

        for ifo in M116_DETECTORS
    }

    return {
        "waveform":
            waveform,

        "projection":
            projection,

        "windowed":
            windowed,

        "segments":
            segments,
    }

In [25]:
def build_gaussian_processing_noise_c3_M116(
    *,
    psds_proc,
    processing_start,
    detector_seeds,
    injector,
):
    noises = {}

    for ifo in M116_DETECTORS:

        noise = (
            gaussian.noise_from_psd(
                psd=(
                    psds_proc[
                        ifo
                    ]
                ),
                length=(
                    M116_C2_CONFIG
                    .processing_length
                ),
                delta_t=(
                    M116_C2_CONFIG
                    .delta_t
                ),
                seed=int(
                    detector_seeds[
                        ifo
                    ]
                ),
            )
        )

        noises[ifo] = (
            injector.set_strain_start_time(
                strain=noise,
                start_time=(
                    processing_start
                ),
                expected_length=(
                    M116_C2_CONFIG
                    .processing_length
                ),
            )
        )

    return noises

In [26]:
def stack_and_normalize_network_c3_M116(
    network,
):
    X_raw = np.stack(
        [
            np.asarray(
                network[
                    ifo
                ]
            )
            for ifo
            in M116_DETECTORS
        ],
        axis=0,
    )

    if (
        X_raw.shape
        !=
        (
            3,
            M116_C2_CONFIG.length,
        )
    ):
        raise ValueError(
            "Unexpected processed network shape: "
            f"{X_raw.shape}"
        )

    if not np.all(
        np.isfinite(
            X_raw
        )
    ):
        raise ValueError(
            "Processed network contains "
            "non-finite values."
        )

    X = (
        normalize_input_per_sample_per_detector_zscore(
            X_raw,
            eps=(
                M116_M10_INPUT_ZSCORE_EPS
            ),
        )
    )

    return X_raw, X

In [27]:
def build_paired_domains_for_manifest_row_M116(
    row,
    *,
    psd_cache,
    strain_cache,
):
    # -------------------------------------------------
    # 1. Source identity and deterministic RNGs
    # -------------------------------------------------

    source_id = str(
        row["source_id"]
    )

    source_index = int(
        row["source_index"]
    )

    placement_rng = np.random.default_rng(
        M116_C3_BUILD_SEED
        +
        M116_C3_PLACEMENT_SEED_OFFSET
        +
        source_index
    )

    gaussian_rng = np.random.default_rng(
        M116_C3_BUILD_SEED
        +
        M116_C3_GAUSSIAN_SEED_OFFSET
        +
        source_index
    )

    injector = SignalInjector(
        config=M116_C2_CONFIG,
        rng=placement_rng,
    )

    gaussian_seeds = {
        ifo:
            int(
                gaussian_rng.integers(
                    0,
                    2**32 - 1,
                )
            )
        for ifo
        in M116_DETECTORS
    }

    # -------------------------------------------------
    # 2. Physical source parameters
    # -------------------------------------------------

    params_ref = CBCParameters(
        mass_1=float(
            row["mass_1"]
        ),
        mass_2=float(
            row["mass_2"]
        ),
        distance=float(
            row[
                "reference_distance_mpc"
            ]
        ),
        inclination=float(
            row["inclination"]
        ),
        ra=float(
            row["ra"]
        ),
        dec=float(
            row["dec"]
        ),
        spin_1z=float(
            row["spin_1z"]
        ),
        spin_2z=float(
            row["spin_2z"]
        ),
        polarization_angle=float(
            row[
                "polarization_angle"
            ]
        ),
    )

    target_snr = float(
        row["target_network_snr"]
    )

    # -------------------------------------------------
    # 3. Assigned physical real-noise window
    # -------------------------------------------------

    processing_start = float(
        row["processing_start"]
    )

    processing_end = float(
        row["processing_end"]
    )

    final_start = (
        processing_start
        +
        M116_C2_CONFIG
        .processing_context_start_seconds
    )

    final_end = (
        final_start
        +
        M116_C2_CONFIG.duration
    )

    final_center = (
        0.5
        *
        (
            final_start
            +
            final_end
        )
    )

    if not np.isclose(
        processing_end
        -
        processing_start,
        M116_C2_CONFIG
        .processing_duration,
    ):
        raise ValueError(
            f"{source_id}: invalid processing duration."
        )

    # -------------------------------------------------
    # 4. Reference waveform and provisional projection
    # -------------------------------------------------

    waveform_ref = (
        waveform_generator_M116.generate(
            params_ref
        )
    )

    projection_provisional = (
        detector_projector_M116.project(
            h_plus=(
                waveform_ref.h_plus
            ),
            h_cross=(
                waveform_ref.h_cross
            ),
            parameters=params_ref,
            geocentric_coalescence_time=(
                final_center
            ),
        )
    )

    windowed_provisional = (
        window_selector_M116.select(
            projected_strains=(
                projection_provisional
                .strains
            ),
            max_duration=(
                M116_C2_CONFIG.duration
            ),
        )
    )

    # -------------------------------------------------
    # 5. Random-contained placement
    # -------------------------------------------------

    abstract_placement = (
        injector
        .choose_segment_placement_containing_network(
            signals=(
                windowed_provisional
                .strains
            ),
            placement_policy=(
                "random_contained"
            ),
            safe_margin_start=float(
                M116_C2_CONFIG
                .safe_margin_start
            ),
            safe_margin_end=float(
                M116_C2_CONFIG
                .safe_margin_end
            ),
            enforce_safe_margins=True,
        )
    )

    placement_shift = (
        final_start
        -
        float(
            abstract_placement
            .segment_start_time
        )
    )

    geocentric_time = (
        final_center
        +
        placement_shift
    )

    placement_offset_s = (
        geocentric_time
        -
        final_center
    )

    # -------------------------------------------------
    # 6. Final reference projection
    # -------------------------------------------------

    projection_ref = (
        detector_projector_M116.project(
            h_plus=(
                waveform_ref.h_plus
            ),
            h_cross=(
                waveform_ref.h_cross
            ),
            parameters=params_ref,
            geocentric_coalescence_time=(
                geocentric_time
            ),
        )
    )

    windowed_ref = (
        window_selector_M116.select(
            projected_strains=(
                projection_ref.strains
            ),
            max_duration=(
                M116_C2_CONFIG.duration
            ),
        )
    )

    tol = (
        2.0
        *
        M116_C2_CONFIG.delta_t
    )

    if (
        windowed_ref
        .metadata
        .used_window_start_time
        <
        final_start - tol
    ):
        raise ValueError(
            f"{source_id}: signal starts outside final segment."
        )

    if (
        windowed_ref
        .metadata
        .used_window_end_time
        >
        final_end + tol
    ):
        raise ValueError(
            f"{source_id}: signal ends outside final segment."
        )

    # -------------------------------------------------
    # 7. Reference signal-only final 4 s
    # -------------------------------------------------

    zero_final = {
        ifo:
            injector.build_zero_strain(
                start_time=final_start,
                length=(
                    M116_C2_CONFIG.length
                ),
            )
        for ifo
        in M116_DETECTORS
    }

    signal_only_ref_results = (
        injector.inject_network(
            noises=zero_final,
            signals=(
                windowed_ref.strains
            ),
        )
    )

    signal_segments_ref = {
        ifo:
            signal_only_ref_results[
                ifo
            ].strain
        for ifo
        in M116_DETECTORS
    }

    # -------------------------------------------------
    # 8. Load real environment and empirical PSDs
    # -------------------------------------------------

    file_group_id = int(
        row["file_group_id"]
    )

    long_strains = (
        get_file_group_strains_c3_M116(
            file_group_id,
            strain_cache=(
                strain_cache
            ),
        )
    )

    empirical_psds = (
        get_block_psds_c3_M116(
            row,
            long_strains=(
                long_strains
            ),
            psd_cache=(
                psd_cache
            ),
        )
    )

    psds_emp_snr = (
        empirical_psds["snr"]
    )

    psds_emp_proc = (
        empirical_psds["proc"]
    )

    # -------------------------------------------------
    # 9. Reference SNR under each PSD domain
    # -------------------------------------------------

    snrs_G0_ref, snr_G0_ref = (
        compute_network_optimal_snr(
            signal_segments=(
                signal_segments_ref
            ),
            psds=(
                psds_G0_snr_M116
            ),
            config=(
                M116_C2_CONFIG
            ),
        )
    )

    snrs_emp_ref, snr_emp_ref = (
        compute_network_optimal_snr(
            signal_segments=(
                signal_segments_ref
            ),
            psds=(
                psds_emp_snr
            ),
            config=(
                M116_C2_CONFIG
            ),
        )
    )

    # -------------------------------------------------
    # 10. Domain-specific distance rescaling
    # -------------------------------------------------

    distance_G0 = (
        rescale_distance_for_target_network_snr(
            current_distance=(
                params_ref.distance
            ),
            current_network_snr=(
                snr_G0_ref
            ),
            target_network_snr=(
                target_snr
            ),
        )
    )

    distance_emp = (
        rescale_distance_for_target_network_snr(
            current_distance=(
                params_ref.distance
            ),
            current_network_snr=(
                snr_emp_ref
            ),
            target_network_snr=(
                target_snr
            ),
        )
    )

    params_G0 = (
        params_ref.with_distance(
            distance_G0
        )
    )

    params_emp = (
        params_ref.with_distance(
            distance_emp
        )
    )

    # -------------------------------------------------
    # 11. Build final domain-specific signals
    # -------------------------------------------------

    signal_G0 = (
        build_signal_network_for_params_c3_M116(
            params_G0,
            geocentric_time=(
                geocentric_time
            ),
            final_start=(
                final_start
            ),
            injector=injector,
        )
    )

    signal_emp = (
        build_signal_network_for_params_c3_M116(
            params_emp,
            geocentric_time=(
                geocentric_time
            ),
            final_start=(
                final_start
            ),
            injector=injector,
        )
    )

    # -------------------------------------------------
    # 12. Final optimal-SNR validation
    # -------------------------------------------------

    snrs_G0_final, snr_G0_final = (
        compute_network_optimal_snr(
            signal_segments=(
                signal_G0[
                    "segments"
                ]
            ),
            psds=(
                psds_G0_snr_M116
            ),
            config=(
                M116_C2_CONFIG
            ),
        )
    )

    snrs_emp_final, snr_emp_final = (
        compute_network_optimal_snr(
            signal_segments=(
                signal_emp[
                    "segments"
                ]
            ),
            psds=(
                psds_emp_snr
            ),
            config=(
                M116_C2_CONFIG
            ),
        )
    )

    validate_snr_rescaling(
        final_network_snr=(
            snr_G0_final
        ),
        target_network_snr=(
            target_snr
        ),
        relative_tolerance=(
            M116_C2_CONFIG
            .snr_relative_tolerance
        ),
    )

    validate_snr_rescaling(
        final_network_snr=(
            snr_emp_final
        ),
        target_network_snr=(
            target_snr
        ),
        relative_tolerance=(
            M116_C2_CONFIG
            .snr_relative_tolerance
        ),
    )

    # -------------------------------------------------
    # 13. G0 and G1 Gaussian processing noise
    # -------------------------------------------------

    noise_G0 = (
        build_gaussian_processing_noise_c3_M116(
            psds_proc=(
                psds_G0_proc_M116
            ),
            processing_start=(
                processing_start
            ),
            detector_seeds=(
                gaussian_seeds
            ),
            injector=injector,
        )
    )

    noise_G1 = (
        build_gaussian_processing_noise_c3_M116(
            psds_proc=(
                psds_emp_proc
            ),
            processing_start=(
                processing_start
            ),
            detector_seeds=(
                gaussian_seeds
            ),
            injector=injector,
        )
    )

    # -------------------------------------------------
    # 14. Exact R processing context
    # -------------------------------------------------

    noise_R = {
        ifo:
            extract_exact_processing_context_M116(
                long_strains[
                    ifo
                ],
                start_time=(
                    processing_start
                ),
                expected_length=(
                    M116_C2_CONFIG
                    .processing_length
                ),
            )
        for ifo
        in M116_DETECTORS
    }

    # -------------------------------------------------
    # 15. Full-projection signal injection
    # -------------------------------------------------

    injected_G0_results = (
        injector.inject_network(
            noises=noise_G0,
            signals=(
                signal_G0[
                    "projection"
                ].strains
            ),
        )
    )

    injected_G1_results = (
        injector.inject_network(
            noises=noise_G1,
            signals=(
                signal_emp[
                    "projection"
                ].strains
            ),
        )
    )

    injected_R_results = (
        injector.inject_network(
            noises=noise_R,
            signals=(
                signal_emp[
                    "projection"
                ].strains
            ),
        )
    )

    injected_G0 = {
        ifo:
            injected_G0_results[
                ifo
            ].strain
        for ifo
        in M116_DETECTORS
    }

    injected_G1 = {
        ifo:
            injected_G1_results[
                ifo
            ].strain
        for ifo
        in M116_DETECTORS
    }

    injected_R = {
        ifo:
            injected_R_results[
                ifo
            ].strain
        for ifo
        in M116_DETECTORS
    }

    # -------------------------------------------------
    # 16. Common processing
    # -------------------------------------------------

    processed_G0 = (
        processor_M116.process_network(
            strains=injected_G0,
            psds=(
                psds_G0_proc_M116
            ),
        )
    )

    processed_G1 = (
        processor_M116.process_network(
            strains=injected_G1,
            psds=(
                psds_emp_proc
            ),
        )
    )

    processed_R = (
        processor_M116.process_network(
            strains=injected_R,
            psds=(
                psds_emp_proc
            ),
        )
    )

    # -------------------------------------------------
    # 17. Stack + M10 z-score
    # -------------------------------------------------

    X_G0_raw, X_G0 = (
        stack_and_normalize_network_c3_M116(
            processed_G0
        )
    )

    X_G1_raw, X_G1 = (
        stack_and_normalize_network_c3_M116(
            processed_G1
        )
    )

    X_R_raw, X_R = (
        stack_and_normalize_network_c3_M116(
            processed_R
        )
    )

    # -------------------------------------------------
    # 18. Final hard contracts
    # -------------------------------------------------

    expected_shape = (
        len(M116_DETECTORS),
        M116_C2_CONFIG.length,
    )

    for domain_name, X in [
        ("G0", X_G0),
        ("G1", X_G1),
        ("R", X_R),
    ]:
        if X.shape != expected_shape:
            raise ValueError(
                f"{source_id} {domain_name}: "
                f"shape={X.shape}, "
                f"expected={expected_shape}."
            )

        if not np.all(
            np.isfinite(X)
        ):
            raise ValueError(
                f"{source_id} {domain_name}: "
                "non-finite final input."
            )

    if not np.isfinite(
        distance_emp
    ):
        raise RuntimeError(
            f"{source_id}: invalid empirical distance."
        )

    # -------------------------------------------------
    # 19. Diagnostics
    # -------------------------------------------------

    raw_noise_std = {}

    for domain_name, network in [
        ("G0", noise_G0),
        ("G1", noise_G1),
        ("R", noise_R),
    ]:
        raw_noise_std[
            domain_name
        ] = {
            ifo:
                float(
                    np.std(
                        np.asarray(
                            network[ifo]
                        )
                    )
                )
            for ifo
            in M116_DETECTORS
        }

    processed_std = {}

    for domain_name, X_raw in [
        ("G0", X_G0_raw),
        ("G1", X_G1_raw),
        ("R", X_R_raw),
    ]:
        processed_std[
            domain_name
        ] = {
            ifo:
                float(
                    X_raw[i].std()
                )
            for i, ifo
            in enumerate(
                M116_DETECTORS
            )
        }

    diagnostics = {
        "source_id":
            source_id,

        "source_index":
            source_index,

        "file_group_id":
            file_group_id,

        "block_id":
            str(
                row["block_id"]
            ),

        "noise_crop_id":
            str(
                row["noise_crop_id"]
            ),

        "chirp_mass":
            float(
                params_ref.chirp_mass
            ),

        "total_mass":
            float(
                params_ref.total_mass
            ),

        "chi_eff":
            float(
                params_ref.chi_eff
            ),

        "target_network_snr":
            target_snr,

        "geocentric_time":
            float(
                geocentric_time
            ),

        "placement_offset_s":
            float(
                placement_offset_s
            ),

        "full_network_duration":
            float(
                windowed_ref
                .metadata
                .full_network_duration
            ),

        "required_final_duration":
            float(
                windowed_ref
                .metadata
                .required_available_final_duration
            ),

        "is_truncated":
            bool(
                windowed_ref
                .metadata
                .is_truncated
            ),

        "distance_G0":
            float(
                distance_G0
            ),

        "distance_G1":
            float(
                distance_emp
            ),

        "distance_R":
            float(
                distance_emp
            ),

        "snr_G0":
            float(
                snr_G0_final
            ),

        "snr_G1":
            float(
                snr_emp_final
            ),

        "snr_R":
            float(
                snr_emp_final
            ),

        "snrs_G0_det":
            dict(
                snrs_G0_final
            ),

        "snrs_G1_det":
            dict(
                snrs_emp_final
            ),

        "snrs_R_det":
            dict(
                snrs_emp_final
            ),

        "raw_noise_std":
            raw_noise_std,

        "processed_std":
            processed_std,

        "all_finite":
            True,
    }

    return {
        "source_id":
            source_id,

        "G0": {
            "X":
                X_G0,
            "X_raw":
                X_G0_raw,
            "distance_mpc":
                float(
                    distance_G0
                ),
            "network_snr":
                float(
                    snr_G0_final
                ),
            "detector_snrs":
                dict(
                    snrs_G0_final
                ),
        },

        "G1": {
            "X":
                X_G1,
            "X_raw":
                X_G1_raw,
            "distance_mpc":
                float(
                    distance_emp
                ),
            "network_snr":
                float(
                    snr_emp_final
                ),
            "detector_snrs":
                dict(
                    snrs_emp_final
                ),
        },

        "R": {
            "X":
                X_R,
            "X_raw":
                X_R_raw,
            "distance_mpc":
                float(
                    distance_emp
                ),
            "network_snr":
                float(
                    snr_emp_final
                ),
            "detector_snrs":
                dict(
                    snrs_emp_final
                ),
        },

        "diagnostics":
            diagnostics,
    }

In [28]:
def get_block_psds_c3_M116(
    row,
    *,
    long_strains,
    psd_cache,
):
    block_id = str(
        row["block_id"]
    )

    if block_id in psd_cache:
        return psd_cache[
            block_id
        ]

    psd_start = float(
        row["psd_start"]
    )

    psd_end = float(
        row["psd_end"]
    )

    psds = {
        "snr": {},
        "proc": {},
    }

    for ifo in M116_DETECTORS:

        result = (
            estimate_block_psds_for_detector_M116(
                strain=long_strains[ifo],
                psd_start=psd_start,
                psd_end=psd_end,
            )
        )

        psds["snr"][ifo] = result["snr"]
        psds["proc"][ifo] = result["proc"]

    psd_cache[
        block_id
    ] = psds

    return psds

In [29]:
def get_file_group_strains_c3_M116(
    file_group_id,
    *,
    strain_cache,
):
    file_group_id = int(
        file_group_id
    )

    if file_group_id not in strain_cache:
        raise KeyError(
            "file_group_id not loaded in the "
            f"local streaming cache: {file_group_id}"
        )

    return strain_cache[
        file_group_id
    ]

In [30]:
def write_domain_sample_M117(
    h5,
    *,
    output_index,
    manifest_row,
    result,
    domain,
):
    domain_result = (
        result[domain]
    )

    diagnostics = (
        result["diagnostics"]
    )

    X = np.asarray(
        domain_result["X"],
        dtype=np.float32,
    )

    expected_shape = (
        3,
        16384,
    )

    if X.shape != expected_shape:
        raise ValueError(
            f"{domain}: unexpected X shape "
            f"{X.shape}"
        )

    if not np.all(
        np.isfinite(X)
    ):
        raise ValueError(
            f"{domain}: X contains "
            "non-finite values."
        )

    # -----------------------------------------
    # Model input
    # -----------------------------------------

    h5["X"][
        output_index
    ] = X

    # -----------------------------------------
    # Physical labels
    # -----------------------------------------

    h5["y"][
        output_index
    ] = np.asarray(
        [
            diagnostics[
                "chirp_mass"
            ],
            diagnostics[
                "total_mass"
            ],
            diagnostics[
                "chi_eff"
            ],
        ],
        dtype=np.float32,
    )

    # -----------------------------------------
    # Identity
    # -----------------------------------------

    h5["source_id"][
        output_index
    ] = str(
        diagnostics[
            "source_id"
        ]
    )

    h5["manifest_index"][
        output_index
    ] = int(
        manifest_row[
            "manifest_index"
        ]
    )

    h5["split"][
        output_index
    ] = str(
        manifest_row[
            "split"
        ]
    )

    # -----------------------------------------
    # SNR / distance
    # -----------------------------------------

    h5[
        "target_network_snr"
    ][
        output_index
    ] = float(
        diagnostics[
            "target_network_snr"
        ]
    )

    h5[
        "final_network_snr"
    ][
        output_index
    ] = float(
        domain_result[
            "network_snr"
        ]
    )

    h5[
        "distance_mpc"
    ][
        output_index
    ] = float(
        domain_result[
            "distance_mpc"
        ]
    )

    detector_snrs = (
        domain_result[
            "detector_snrs"
        ]
    )

    for ifo in [
        "H1",
        "L1",
        "V1",
    ]:
        h5[
            f"snr_{ifo}"
        ][
            output_index
        ] = float(
            detector_snrs[
                ifo
            ]
        )

    # -----------------------------------------
    # Geometry
    # -----------------------------------------

    h5[
        "geocentric_time"
    ][
        output_index
    ] = float(
        diagnostics[
            "geocentric_time"
        ]
    )

    h5[
        "placement_offset_s"
    ][
        output_index
    ] = float(
        diagnostics[
            "placement_offset_s"
        ]
    )

    h5[
        "full_network_duration"
    ][
        output_index
    ] = float(
        diagnostics[
            "full_network_duration"
        ]
    )

    h5[
        "required_final_duration"
    ][
        output_index
    ] = float(
        diagnostics[
            "required_final_duration"
        ]
    )

    # -----------------------------------------
    # Environment
    # -----------------------------------------

    h5[
        "file_group_id"
    ][
        output_index
    ] = int(
        diagnostics[
            "file_group_id"
        ]
    )

    h5[
        "block_id"
    ][
        output_index
    ] = str(
        diagnostics[
            "block_id"
        ]
    )

    h5[
        "noise_crop_id"
    ][
        output_index
    ] = str(
        diagnostics[
            "noise_crop_id"
        ]
    )

    h5[
        "is_truncated"
    ][
        output_index
    ] = bool(
        diagnostics[
            "is_truncated"
        ]
    )

    # Mark complete only after every
    # field has been written.
    h5[
        "status"
    ][
        output_index
    ] = np.uint8(1)

(Para localizar/cargar los file_group_id)

In [31]:
def load_file_group_strains_M116(
    file_group_row,
):
    group_id = int(
        file_group_row[
            "file_group_id"
        ]
    )

    strains = {}

    for ifo in M116_DETECTORS:

        url = file_group_row[
            f"{ifo}_url"
        ]

        local_path = (
            download_if_needed(
                url=url,
                cache_dir=GWOSC_CACHE_M116,
            )
        )

        ts = (
            read_gwosc_hdf5_as_pycbc_timeseries(
                local_path
            )
        )

        strains[ifo] = ts

    if set(
        strains.keys()
    ) != set(
        M116_DETECTORS
    ):
        raise RuntimeError(
            f"Incomplete detector set for "
            f"file_group_id={group_id}: "
            f"{list(strains)}"
        )

    return strains

In [32]:
FILE_GROUPS_PATH = (
    DATA_ROOT
    / "processed"
    / "m11_6_manifests"
    / "m11_6_file_groups.csv"
)

df_file_groups_M117 = pd.read_csv(
    FILE_GROUPS_PATH
)

In [33]:
#Selecciona group
pilot_group_meta_M117 = (
    df_file_groups_M117[
        df_file_groups_M117[
            "file_group_id"
        ]
        == PILOT_FILE_GROUP_ID
    ]
)

assert len(
    pilot_group_meta_M117
) == 1

pilot_group_meta_M117 = (
    pilot_group_meta_M117.iloc[0]
)

pilot_anchor_event_M117 = str(
    pilot_group_meta_M117[
        "anchor_event"
    ]
)

print(
    "Pilot file_group_id:",
    PILOT_FILE_GROUP_ID,
)

print(
    "Anchor event:",
    pilot_anchor_event_M117,
)

Pilot file_group_id: 6
Anchor event: GW191204_110529


Cargamos el grupo piloto

In [34]:
#Resolve URLs (light)
events_raw_M117 = (
    fetch_gwosc_catalog_events(
        catalog=M116_CATALOG,
        include_default_parameters=True,
    )
)

events_df_M117 = (
    gwosc_events_to_parameter_df(
        events_raw_M117,
        catalog_name=M116_CATALOG,
    )
)

In [35]:
pilot_anchor_df_M117 = (
    events_df_M117[
        events_df_M117[
            "event"
        ]
        == pilot_anchor_event_M117
    ]
    .copy()
)

assert len(
    pilot_anchor_df_M117
) == 1

In [36]:
pilot_urls_M117, \
pilot_failed_urls_M117 = (
    build_gwosc_urls_for_events(
        pilot_anchor_df_M117,
        catalog=M116_CATALOG,
        sample_rate=M116_SAMPLE_RATE,
    )
)

assert len(
    pilot_failed_urls_M117
) == 0

In [37]:
#Build row for loader
pilot_urls_for_event_M117 = (
    pilot_urls_M117[
        pilot_anchor_event_M117
    ]
)

pilot_group_row_M117 = pd.Series({
    "file_group_id":
        PILOT_FILE_GROUP_ID,

    "H1_url":
        pilot_urls_for_event_M117["H1"],

    "L1_url":
        pilot_urls_for_event_M117["L1"],

    "V1_url":
        pilot_urls_for_event_M117["V1"],
})

In [38]:
#Ahora load el only environment
pilot_long_strains = (
    load_file_group_strains_M116(
        pilot_group_row_M117
    )
)

Using cached file: H-H1_GWOSC_4KHZ_R1-1259490700-4096.hdf5
Using cached file: L-L1_GWOSC_4KHZ_R1-1259490700-4096.hdf5
Using cached file: V-V1_GWOSC_O3b_4KHZ_R1-1259491328-4096.hdf5


In [39]:
for ifo in M116_DETECTORS:
    print(
        ifo,
        "samples:",
        len(
            pilot_long_strains[ifo]
        ),
        "duration:",
        float(
            pilot_long_strains[ifo].duration
        ),
    )

H1 samples: 16777216 duration: 4096.0
L1 samples: 16777216 duration: 4096.0
V1 samples: 16777216 duration: 4096.0


In [40]:
#Local caches
pilot_psd_cache_M117 = {}

pilot_strain_cache_M117 = {
    PILOT_FILE_GROUP_ID:
        pilot_long_strains
}

In [41]:
for domain, path in (
    pilot_h5_paths.items()
):

    if path.exists():
        path.unlink()

    create_pilot_hdf5(
        path,
        n_samples=PILOT_N,
        domain=domain,
    )

In [42]:
pilot_h5 = {
    domain:
        h5py.File(
            path,
            "r+",
        )
    for domain, path
    in pilot_h5_paths.items()
}

In [43]:
generation_log_M117 = []

for output_index, row in (
    df_pilot.iterrows()
):

    source_id = str(
        row["source_id"]
    )

    print(
        f"[{output_index + 1}/"
        f"{len(df_pilot)}] "
        f"{source_id}",
        end="  ",
        flush=True,
    )

    try:

        result = (
            build_paired_domains_for_manifest_row_M116(
                row,
                psd_cache=(
                    pilot_psd_cache_M117
                ),
                strain_cache=(
                    pilot_strain_cache_M117
                ),
            )
        )

        # Paired contract before writing.
        assert np.isclose(
            result["G1"][
                "distance_mpc"
            ],
            result["R"][
                "distance_mpc"
            ],
            rtol=0.0,
            atol=1e-10,
        )

        for domain in [
            "G0",
            "G1",
            "R",
        ]:

            write_domain_sample_M117(
                pilot_h5[
                    domain
                ],
                output_index=(
                    output_index
                ),
                manifest_row=row,
                result=result,
                domain=domain,
            )

        # Flush after the complete paired triplet.
        for h5 in (
            pilot_h5.values()
        ):
            h5.flush()

        generation_log_M117.append({
            "source_id":
                source_id,

            "status":
                "PASS",

            "error":
                "",
        })

        print("PASS")

        # Do not retain generated arrays.
        del result

    except Exception as exc:

        generation_log_M117.append({
            "source_id":
                source_id,

            "status":
                "FAIL",

            "error":
                (
                    f"{type(exc).__name__}: "
                    f"{exc}"
                ),
        })

        print(
            "FAIL",
            type(exc).__name__,
            str(exc),
        )

        # Pilot behaviour: fail immediately.
        raise

    finally:

        gc.collect()

[1/5] M116_SRC_005444  PASS
[2/5] M116_SRC_010223  PASS
[3/5] M116_SRC_006138  PASS
[4/5] M116_SRC_008483  PASS
[5/5] M116_SRC_016123  PASS


In [44]:
for h5 in (
    pilot_h5.values()
):
    h5.close()

del pilot_h5

gc.collect()

print(
    "Pilot HDF5 files closed."
)

Pilot HDF5 files closed.


In [45]:
print(
    "Local PSD cache blocks:",
    len(
        pilot_psd_cache_M117
    ),
)

Local PSD cache blocks: 1


In [46]:
del pilot_long_strains
del pilot_strain_cache_M117
del pilot_psd_cache_M117

gc.collect()

print(
    "Real HLV environment and "
    "local PSD cache released."
)

Real HLV environment and local PSD cache released.


In [47]:
reloaded_pilot_M117 = {}

for domain, path in (
    pilot_h5_paths.items()
):

    with h5py.File(
        path,
        "r",
    ) as h5:

        reloaded_pilot_M117[
            domain
        ] = {
            "X":
                h5["X"][:],

            "y":
                h5["y"][:],

            "source_id":
                h5[
                    "source_id"
                ][:].astype(str),

            "manifest_index":
                h5[
                    "manifest_index"
                ][:],

            "status":
                h5[
                    "status"
                ][:],

            "distance_mpc":
                h5[
                    "distance_mpc"
                ][:],

            "target_network_snr":
                h5[
                    "target_network_snr"
                ][:],

            "final_network_snr":
                h5[
                    "final_network_snr"
                ][:],
        }

In [48]:
for domain in [
    "G0",
    "G1",
    "R",
]:

    data = (
        reloaded_pilot_M117[
            domain
        ]
    )

    assert data[
        "X"
    ].shape == (
        5,
        3,
        16384,
    )

    assert np.all(
        np.isfinite(
            data["X"]
        )
    )

    assert np.all(
        data[
            "status"
        ] == 1
    )

    assert np.array_equal(
        data[
            "manifest_index"
        ],
        df_pilot[
            "manifest_index"
        ].to_numpy(),
    )

    assert np.array_equal(
        data[
            "source_id"
        ],
        df_pilot[
            "source_id"
        ].astype(str).to_numpy(),
    )

print(
    "Per-domain persistence contracts passed."
)

Per-domain persistence contracts passed.


In [49]:
for domain in [
    "G1",
    "R",
]:

    assert np.array_equal(
        reloaded_pilot_M117[
            "G0"
        ][
            "source_id"
        ],
        reloaded_pilot_M117[
            domain
        ][
            "source_id"
        ],
    )

    assert np.array_equal(
        reloaded_pilot_M117[
            "G0"
        ][
            "manifest_index"
        ],
        reloaded_pilot_M117[
            domain
        ][
            "manifest_index"
        ],
    )

print(
    "Cross-domain paired identity passed."
)

Cross-domain paired identity passed.


In [50]:
assert np.allclose(
    reloaded_pilot_M117[
        "G1"
    ][
        "distance_mpc"
    ],
    reloaded_pilot_M117[
        "R"
    ][
        "distance_mpc"
    ],
    rtol=0.0,
    atol=1e-10,
)

print(
    "Persisted G1/R distance "
    "pairing passed."
)

Persisted G1/R distance pairing passed.


In [51]:
snr_rows_M117 = []

for domain in [
    "G0",
    "G1",
    "R",
]:

    data = (
        reloaded_pilot_M117[
            domain
        ]
    )

    for i in range(
        PILOT_N
    ):

        target = float(
            data[
                "target_network_snr"
            ][i]
        )

        final = float(
            data[
                "final_network_snr"
            ][i]
        )

        snr_rows_M117.append({
            "domain":
                domain,

            "source_id":
                data[
                    "source_id"
                ][i],

            "target_snr":
                target,

            "final_snr":
                final,

            "relative_error":
                abs(
                    final
                    -
                    target
                )
                /
                target,
        })

df_snr_persistence_M117 = (
    pd.DataFrame(
        snr_rows_M117
    )
)

display(
    df_snr_persistence_M117
)

,domain,source_id,target_snr,final_snr,relative_error
0,G0,M116_SRC_005444,18.602381,18.602381,1.909817e-16
1,G0,M116_SRC_010223,15.458726,15.458726,1.149097e-16
2,G0,M116_SRC_006138,15.247832,15.247832,5.824949e-16
3,G0,M116_SRC_008483,15.877320,15.877320,5.594007e-16
4,G0,M116_SRC_016123,12.155867,12.155867,1.899711e-15
5,G1,M116_SRC_005444,18.602381,18.602381,0.000000e+00
6,G1,M116_SRC_010223,15.458726,15.458726,1.149097e-16
7,G1,M116_SRC_006138,15.247832,15.247832,1.164990e-15
8,G1,M116_SRC_008483,15.877320,15.877320,3.356404e-16
9,G1,M116_SRC_016123,12.155867,12.155867,3.361028e-15


In [52]:
zscore_rows_M117 = []

for domain in [
    "G0",
    "G1",
    "R",
]:

    X = (
        reloaded_pilot_M117[
            domain
        ][
            "X"
        ]
    )

    for i in range(
        len(X)
    ):

        for j, ifo in enumerate([
            "H1",
            "L1",
            "V1",
        ]):

            zscore_rows_M117.append({
                "domain":
                    domain,

                "source_index":
                    i,

                "detector":
                    ifo,

                "mean":
                    float(
                        X[
                            i,
                            j,
                        ].mean()
                    ),

                "std":
                    float(
                        X[
                            i,
                            j,
                        ].std()
                    ),
            })

df_zscore_persistence_M117 = (
    pd.DataFrame(
        zscore_rows_M117
    )
)

display(
    df_zscore_persistence_M117
)

,domain,source_index,detector,mean,std
0,G0,0,H1,3.201421e-10,1.0
1,G0,0,L1,2.910383e-10,1.0
2,G0,0,V1,5.529728e-10,1.0
3,G0,1,H1,1.222361e-09,1.0
4,G0,1,L1,1.600711e-10,1.0
5,G0,1,V1,-1.455192e-10,1.0
6,G0,2,H1,5.238689e-10,1.0
7,G0,2,L1,5.238689e-10,1.0
8,G0,2,V1,1.018634e-10,1.0
9,G0,3,H1,5.093170e-10,1.0


In [53]:
for domain, path in (
    pilot_h5_paths.items()
):

    print(
        domain,
        f"{path.stat().st_size / 1024**2:.2f} MB",
    )

G0 0.96 MB
G1 0.96 MB
R 0.96 MB


In [54]:
df_generation_log_M117 = (
    pd.DataFrame(
        generation_log_M117
    )
)

display(
    df_generation_log_M117
)

,source_id,status,error
0,M116_SRC_005444,PASS,
1,M116_SRC_010223,PASS,
2,M116_SRC_006138,PASS,
3,M116_SRC_008483,PASS,
4,M116_SRC_016123,PASS,


In [56]:
assert (
    df_generation_log_M117[
        "status"
    ]
    == "PASS"
).all()

assert len(
    df_generation_log_M117
) == 5

## M11.7 streaming pilot — conclusion

The streaming production strategy has now been tested independently of the
large M11.6 validation notebook.

For one real HLV environment:

1. the environment was loaded once;
2. empirical PSDs were cached only locally by block;
3. five paired sources were generated as G0/G1/R;
4. each triplet was written immediately to disk;
5. no generated sample collection was accumulated in memory;
6. the real strain and PSD cache were explicitly released;
7. the three HDF5 files were reopened from disk;
8. paired source identity, tensor geometry, finiteness, SNR control,
   G1/R distance equality and M10 input normalization were preserved.

This validates the execution strategy required for full M11.6 dataset
production.

The next implementation step is to promote this validated notebook workflow
to a standalone streaming generation script operating sequentially over all
`file_group_id` environments.